In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

### Make a specific spatial region spatially unfair

**TODO.**

The idea is to partition the space with a uniform grid, select a specific cell, and then relabel the objects associated with that cell. This simulates a trajectory classifier that is unfair w.r.t. that cell's objects.

In [ ]:
# Read the US Census traits
path_atlanta_tracts = './data_simulator/huge_dataset/atlanta_census_tracts_2020.geojson'
atlanta_tracts = gpd.read_file(path_atlanta_tracts)['geometry'].to_frame()
atlanta_tracts

Plot a simple map with the Atlanta 2020 US Census Tracts.

In [ ]:
import folium

# Base map centered on candidates
minx, miny, maxx, maxy = atlanta_tracts.total_bounds
m = folium.Map(location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=12, prefer_canvas=True)
folium.GeoJson(atlanta_tracts).add_to(m)

m

# Map the users' stop segments to the traits

In [ ]:
# Read the dataframe with the users' stop segments.
path_stop_df = './data_simulator/huge_dataset/dataset_simulator_trajectories.compressed.parquet.stops.parquet'
stop_df = pd.read_parquet(path_stop_df)
stop_df = gpd.GeoDataFrame(stop_df, 
                           geometry=gpd.points_from_xy(stop_df.lng, stop_df.lat), 
                           crs="EPSG:4326").loc[:, ['uid', 'geometry']]
stop_df


# Associated each stop segment to a tract via its centroid.
mapped_stops = stop_df.sjoin(atlanta_tracts,
                             how="left",
                             predicate="within")[['uid', 'index_right']]
mapped_stops

In [ ]:
# For each tract, count the number of users with at least a stop in that tract.
count_users_per_tract = mapped_stops.drop_duplicates()['index_right'].value_counts()

# Compute the frequency of each pair '(user, tract)'.
count_user_tract_frequency = mapped_stops.groupby(['uid', 'index_right']).size()
# count_user_tract_frequency.loc[count_user_tract_frequency >= 5]

display(count_users_per_tract)
display(count_user_tract_frequency)

In [ ]:
# TODO: IDEA for injecting unfairness => we can pick 1/2/3/4 tracts as seeds, and initially use a negative buffer to make their
# area equal to 0. 
# We then progressively expand their buffers, progressively taking in the stop segments' centroids within them, and thus
# progressively increasing the number of users affected by unfairness.

### Write the synthetic unfair labels to disk